# Preflight: dataset inventory and overlap

This notebook is tailored to this repository's downloader and staged layout (`data/altair`, `data/posh`, and `data/aconcagua`). It works before or after downloads: unavailable files are reported as not staged, and no data are downloaded by this notebook. Use [`01_download_data_started.ipynb`](01_download_data_started.ipynb) or `python scripts/download_data.py` to fetch data.

It inventories profile files and supporting metadata, inspects schemas without loading full matrices into memory, and checks practical overlap. H5AD profiles may be inspected only when present; Parquet schemas are read from their footer. Outputs go to `reports/preflight_inventory/`.

In [ ]:
from pathlib import Path
import json, os, sys
import pandas as pd
from IPython.display import display

# Resolve the repository regardless of whether Jupyter starts in the repo or notebooks/.
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / 'scripts' / 'download_data.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Could not locate scripts/download_data.py. Start Jupyter from this repository or its notebooks folder.')
DATA = Path(os.environ.get('SBI2_DATA_DIR', ROOT / 'data')).expanduser().resolve()
OUT = ROOT / 'reports' / 'preflight_inventory'
OUT.mkdir(parents=True, exist_ok=True)
print('Repository:', ROOT)
print('Data root:', DATA)
print('Reports:', OUT)


## 1. Expected download inventory

The expected files mirror `scripts/download_data.py` and its core preset. A file marked missing can be downloaded with the participant starter or CLI. The OPS0077 H5ADs are extracted from `markers_guide.zip`; they are not separate source downloads.

In [ ]:
expected = [
    ('Altair phase', 'altair/phase_guide.h5ad', 'One phase profile per guide'),
    ('Altair phase', 'altair/phase_guide_x_screen.h5ad', 'One phase profile per guide and screen'),
    ('Altair phase metadata', 'altair/screen_map.csv', 'Screen mapping'),
    ('Altair phase metadata', 'altair/unexpressed_genes.csv', 'Expression context'),
    ('Altair 4i', 'altair/markers_guide.zip', 'Marker archive; required to extract guide profiles'),
    ('Altair 4i metadata', 'altair/markers.csv', 'Marker inventory'),
    ('Altair 4i metadata', 'altair/manifest.csv', 'Prepared release manifest'),
    ('POSH', 'posh/morphology_124.pq', 'Processed morphology profiles'),
    ('Aconcagua', 'aconcagua/interphase_aggregated_data.h5ad', 'HeLa interphase aggregate profile'),
    ('Aconcagua', 'aconcagua/mitotic_aggregated_data.h5ad', 'HeLa mitotic aggregate profile'),
    ('Aconcagua metadata', 'aconcagua/metadata/experimental_metadata.yaml', 'Experiment metadata'),
    ('Aconcagua metadata', 'aconcagua/metadata/feature_definitions.csv', 'Feature definitions'),
    ('Aconcagua metadata', 'aconcagua/metadata/perturbation_library.csv', 'Perturbation/guide library'),
]
markers = ['b-catenin','c-myc','gh2ax','nfkb','p21','p53','prb','ps6','rb','rsp6']
expected += [('Altair Biohub_OPS0077 4i', f'altair/ops0077_4i/{m}_guide_x_screen.h5ad', 'Extracted guide-by-screen profile') for m in markers]
rows=[]
for dataset, rel, content in expected:
    p=DATA/rel
    rows.append({'dataset':dataset,'path':rel,'content':content,'status':'staged' if p.is_file() else 'not staged','size_bytes':p.stat().st_size if p.is_file() else None})
inventory=pd.DataFrame(rows)
display(inventory)
inventory.to_csv(OUT/'expected_file_inventory.csv',index=False)
print(f"Staged {sum(inventory.status == 'staged')} of {len(inventory)} expected files; reports saved under {OUT}")


## 2. Inspect profile schemas and dimensions

H5AD metadata are read in backed mode. Parquet row counts and column names come from file metadata, avoiding a full table read. This section reports missing files rather than failing when participants stage only a subset.

In [ ]:
profile_paths = [DATA/'altair/phase_guide.h5ad', DATA/'altair/phase_guide_x_screen.h5ad', *sorted((DATA/'altair/ops0077_4i').glob('*_guide_x_screen.h5ad')), DATA/'posh/morphology_124.pq', DATA/'aconcagua/interphase_aggregated_data.h5ad', DATA/'aconcagua/mitotic_aggregated_data.h5ad']
schemas=[]
for path in profile_paths:
    if not path.is_file():
        continue
    item={'dataset':str(path.relative_to(DATA)),'bytes':path.stat().st_size,'format':path.suffix.lstrip('.').upper()}
    try:
        if path.suffix == '.h5ad':
            import anndata as ad
            obj=ad.read_h5ad(path,backed='r')
            item.update(rows=int(obj.n_obs),features=int(obj.n_vars),obs_columns=list(map(str,obj.obs.columns)),var_columns=list(map(str,obj.var.columns)))
            item['observation_id_duplicates']=int(obj.obs_names.duplicated().sum())
            item['feature_id_duplicates']=int(obj.var_names.duplicated().sum())
            obj.file.close()
        elif path.suffix == '.pq':
            import pyarrow.parquet as pq
            pf=pq.ParquetFile(path)
            item.update(rows=pf.metadata.num_rows,features=len(pf.schema.names),columns=pf.schema.names,row_groups=pf.metadata.num_row_groups)
    except Exception as e:
        item['inspection_error']=f'{type(e).__name__}: {e}'
    schemas.append(item)
summary=pd.DataFrame([{k:v for k,v in x.items() if k not in ('obs_columns','var_columns','columns')} for x in schemas])
display(summary if len(summary) else pd.DataFrame({'status':['No profile files staged yet. Run the participant download notebook first.']}))
(OUT/'profile_schema_inventory.json').write_text(json.dumps(schemas,indent=2,default=str)+'\n')


## 3. Guide-level overlap: Altair phase summaries versus Biohub_OPS0077 4i

This is the direct within-Altair guide comparison. We compare the 4i profile guide identifiers and target annotations against both `phase_guide.h5ad` (median across screens) and `phase_guide_x_screen.h5ad` (per screen). We check that each shared guide maps to the same target; the per-screen source is collapsed to unique guide identities for this membership check. Aconcagua is intentionally not included here because its downloaded profiles are gene/perturbation aggregates, not guide-level matrices.

In [ ]:
def guide_target_map(path):
    import anndata as ad
    obj=ad.read_h5ad(path,backed='r')
    obs=obj.obs
    guide_col=next((c for c in ['guide','sgRNA','Metadata_sgRNA','barcode','guide_id'] if c in obs.columns),None)
    target_col=next((c for c in ['target_gene','gene','gene_name','Metadata_Gene'] if c in obs.columns),None)
    if guide_col is None or target_col is None:
        obj.file.close(); raise ValueError(f'{path.name}: expected guide/target columns; found {list(obs.columns)}')
    pairs=obs[[guide_col,target_col]].dropna().astype(str)
    grouped=pairs.groupby(guide_col)[target_col].agg(lambda x: sorted(set(x)))
    conflicts=grouped[grouped.map(len)>1]
    mapping={str(g):v[0] for g,v in grouped.items()}
    obj.file.close()
    return mapping, int(len(conflicts))

phase_paths=[DATA/'altair/phase_guide.h5ad',DATA/'altair/phase_guide_x_screen.h5ad']
overlap_rows=[]
marker_paths=sorted((DATA/'altair/ops0077_4i').glob('*_guide_x_screen.h5ad'))
for phase in phase_paths:
    if not phase.is_file():
        print(f'{phase.name} is not staged; its guide overlap is skipped.')
        continue
    phase_map,phase_conflicts=guide_target_map(phase)
    for p in marker_paths:
        marker_map,marker_conflicts=guide_target_map(p)
        shared=set(phase_map)&set(marker_map)
        mismatches=sorted(g for g in shared if phase_map[g].upper()!=marker_map[g].upper())
        overlap_rows.append({'phase_profile':phase.name,'marker_profile':p.name,'phase_guides':len(phase_map),'marker_guides':len(marker_map),'shared_guides':len(shared),'phase_only':len(set(phase_map)-set(marker_map)),'marker_only':len(set(marker_map)-set(phase_map)),'shared_target_mismatches':len(mismatches),'phase_internal_target_conflicts':phase_conflicts,'marker_internal_target_conflicts':marker_conflicts,'mismatch_examples':mismatches[:10]})
guide_overlap=pd.DataFrame(overlap_rows)
display(guide_overlap if len(guide_overlap) else pd.DataFrame({'status':['No phase/OPS0077 guide overlap computed. Confirm that a phase profile and OPS0077 guide profiles are staged.']}))
guide_overlap.to_csv(OUT/'altair_phase_ops0077_guide_overlap.csv',index=False)


## 4. Gene-level overlap across datasets (exploratory)

This section checks identifiers available from profile annotations and Aconcagua's perturbation library. It does not equate feature columns across assays: feature names are assay-specific and should not be concatenated as if they were the same measurements. POSH schemas vary by release; the notebook reports candidate gene/target columns only when their names are recognizable. Review the printed fields before interpreting counts.

In [ ]:
def norm(s):
    return {str(x).strip().upper() for x in s if pd.notna(x) and str(x).strip() and str(x).strip().upper() not in {'NAN','NONE','NULL'}}

def candidate_gene_values(path):
    values=set(); details=[]
    if not path.is_file(): return values,details
    if path.suffix=='.h5ad':
        import anndata as ad
        obj=ad.read_h5ad(path,backed='r'); frame=obj.obs
        candidates=[c for c in frame.columns if str(c).lower() in {'gene','gene_name','target_gene','gene_symbol','aggregate_id','gene_id'}]
        for col in candidates:
            vals=norm(frame[col].dropna())
            if vals: values |= vals; details.append({'file':str(path.relative_to(DATA)),'field':col,'unique_values':len(vals),'examples':sorted(vals)[:5]})
        obj.file.close()
    elif path.suffix=='.pq':
        import pyarrow.parquet as pq
        pf=pq.ParquetFile(path); names=pf.schema.names
        candidates=[c for c in names if c.lower() in {'gene','gene_name','target_gene','gene_symbol','aggregate_id','gene_id'}]
        for col in candidates:
            vals=set()
            for i in range(pf.metadata.num_row_groups): vals |= norm(pf.read_row_group(i,columns=[col]).column(0).to_pylist())
            if vals: values |= vals; details.append({'file':str(path.relative_to(DATA)),'field':col,'unique_values':len(vals),'examples':sorted(vals)[:5]})
    return values,details

profile_gene_sets={}; gene_details=[]
for p in profile_paths:
    if p.is_file():
        vals,info=candidate_gene_values(p)
        if vals: profile_gene_sets[str(p.relative_to(DATA))]=vals; gene_details.extend(info)
lib_path=DATA/'aconcagua/metadata/perturbation_library.csv'
if lib_path.is_file():
    lib=pd.read_csv(lib_path,low_memory=False)
    gene_col=next((c for c in lib.columns if c.lower()=='gene_symbol'),None)
    if gene_col:
        vals=norm(lib[gene_col].dropna()); profile_gene_sets['aconcagua/metadata/perturbation_library.csv']=vals
        gene_details.append({'file':'aconcagua/metadata/perturbation_library.csv','field':gene_col,'unique_values':len(vals),'examples':sorted(vals)[:5]})
display(pd.DataFrame(gene_details) if gene_details else pd.DataFrame({'status':['No recognizable gene/target annotation fields found in staged profiles.']}))
pairs=[]
for i,a in enumerate(profile_gene_sets):
    for b in list(profile_gene_sets)[i+1:]:
        shared=profile_gene_sets[a]&profile_gene_sets[b]
        pairs.append({'dataset_a':a,'dataset_b':b,'genes_a':len(profile_gene_sets[a]),'genes_b':len(profile_gene_sets[b]),'shared_genes':len(shared),'jaccard':len(shared)/len(profile_gene_sets[a]|profile_gene_sets[b]) if profile_gene_sets[a]|profile_gene_sets[b] else 0,'examples':sorted(shared)[:12]})
gene_overlap=pd.DataFrame(pairs).sort_values('shared_genes',ascending=False) if pairs else pd.DataFrame()
display(gene_overlap)
gene_overlap.to_csv(OUT/'candidate_gene_overlap.csv',index=False)


## 5. Interpretation and saved outputs

- **Altair phase vs OPS0077 4i:** guide overlap supports guide-level phenotype comparisons for shared perturbations. It does not establish that the independently summarized feature matrices are directly comparable without normalization and assay-aware analysis.
- **POSH:** same broad A549 context makes target-level comparison plausible where target symbols overlap, but its screen design and processed table schema differ. Confirm row semantics and target mapping before analysis.
- **Aconcagua:** useful for cross-atlas phenotype conservation at shared gene targets, but it is HeLa and aggregate-level; cell-line and assay differences are confounders, and guides cannot be reconstructed from the aggregate profiles.
- **Outputs:** `expected_file_inventory.csv`, `profile_schema_inventory.json`, `altair_phase_ops0077_guide_overlap.csv`, and `candidate_gene_overlap.csv` under `reports/preflight_inventory/`.

For integrity checks and a fuller staged-file report, run `python scripts/run_quality_checks.py --data-dir data --allow-incomplete`. The downloader's manifest records downloaded paths, byte sizes, and checksums.